In [ ]:
import pandas as pd

# === FILE PATHS ===
mapping_file = "D:/Tushar/main_with_subs_only.xlsx"
indent_file  = "D:/PPC Plan/Monthly Indent/Demo.xlsx"

# === LOAD FILES ===
print("Loading mapping...")
df_mapping = pd.read_excel(mapping_file)

print("Loading demand file...")
df_indent = pd.read_excel(indent_file)

# === RENAME COLUMNS ===
df_mapping = df_mapping.rename(columns={
    'Main_Label': 'Child_Part',
    'Sub_Label':  'Switch_Part',
    'Sub_Count':  'Qty_per_Switch'
})

df_indent = df_indent.rename(columns={
    'Part number': 'Switch_Part'
})

# === MONTH COLUMNS FROM DEMO FILE ===
month_cols = [
    "Feb'26 QTY",
    "Mar'26",
    "Apr'26",
    "May'26",
    "Jun'26",
    "Jul'26"
]

# === MERGE ===
df_merged = pd.merge(
    df_mapping[['Child_Part','Switch_Part','Qty_per_Switch']],
    df_indent[['Switch_Part'] + month_cols],
    on='Switch_Part',
    how='left'
)

print("Merge done")

# === CLEAN MONTH NAMES ===
clean_months = [m.replace("'", "").replace(" QTY","") for m in month_cols]

# === CALCULATE REQUIREMENTS ===
for month, clean in zip(month_cols, clean_months):

    daily_col = f"Daily_{clean}"
    two_col   = f"2Days_{clean}"

    df_merged[daily_col] = df_merged[month] / 30.0
    df_merged[two_col]   = df_merged[daily_col] * df_merged['Qty_per_Switch'] * 2

# === AGGREGATE ===
agg_dict = {f"Daily_{c}": 'sum' for c in clean_months}

totals = df_merged.groupby('Child_Part', as_index=False).agg(agg_dict)

for c in clean_months:
    totals[f"2Days_{c}"] = totals[f"Daily_{c}"] * 2

totals.to_excel("Child_Totals_2Days.xlsx", index=False)

print("Saved totals")

# === DETAILS ===
detail_cols = (
    ['Child_Part','Switch_Part','Qty_per_Switch'] +
    month_cols +
    [f"Daily_{c}" for c in clean_months] +
    [f"2Days_{c}" for c in clean_months]
)

details = df_merged[detail_cols]
details.to_excel("Child_Details_2Days.xlsx", index=False)

print("Saved details")
print("✅ Done")
